# ⬡ PRISM · ICU Risk Assessment Dashboard

Dual **sepsis + mortality** risk monitoring · Temporal Fusion Transformer · MIMIC-IV

### Run in order:
1. **Cell 1** — Install packages
2. **Cell 2** — Generate synthetic patient data
3. **Cell 3** — Write dashboard files
4. **Cell 4** — Launch Streamlit + open URL
5. **Cell 5** *(optional)* — Live simulation every 3 seconds
6. **Cell 6** *(optional)* — Save to Google Drive

> ⚠️ Keep this Colab tab open while using the dashboard — closing it stops the server.

In [1]:
# Cell 1 — Install
!pip install streamlit plotly -q
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.3 MB/s eta 0:00:00
Done


In [2]:
# Cell 2 — Generate synthetic patient data
import numpy as np, json, os
np.random.seed(42)
os.makedirs('/content/prism', exist_ok=True)
FEATURE_NAMES = [
    'antibiotic','mbp_ni','urine_output','gcs','totalco2',
    'ventilation','po2_count','sbp_ni','chloride_count','rdw',
    'sodium_count','ptt','lactate','resp_rate_count','hemoglobin',
    'platelet','creatinine','bilirubin','wbc','glucose',
    'heart_rate','sbp','dbp','spo2','resp_rate',
    'temp','ph','base_excess','bicarbonate','aniongap',
    'potassium','sodium','chloride','bun','albumin',
    'fibrinogen','inr','ptt_count','troponin','bnp',
    'lactate_count','po2','pco2','fio2','peep',
    'norepinephrine','vasopressin','dopamine','dobutamine','heparin'
]
DIAGNOSES = ['Respiratory failure','Septic shock','Post-operative monitoring',
    'Cardiac arrest','Pneumonia','UTI','Bowel perforation','Pancreatitis',
    'Liver failure','Renal failure','Trauma','Burns','Stroke','COPD exacerbation','DKA']
BEDS = [f'ICU-{i+1:02d}' for i in range(20)]
N = 200
def gen_traj(base, label):
    v = float(np.clip(base + np.random.normal(0, 0.08), 0.02, 0.95))
    trend = np.random.choice([0.006, 0.012]) if label else np.random.choice([-0.006, 0.001])
    traj = []
    for _ in range(24):
        v = float(np.clip(v + trend + np.random.normal(0, 0.018), 0.02, 0.97))
        traj.append(round(v, 4))
    traj[-1] = round(float(np.clip(base + np.random.normal(0, 0.02), 0.02, 0.97)), 4)
    return traj
def gen_feats(score, n=6):
    w = np.abs(np.random.dirichlet(np.ones(len(FEATURE_NAMES)) * (1 + score * 4)))
    boost = {'antibiotic':0.04,'lactate':0.035,'gcs':0.03,'norepinephrine':0.03,
             'ventilation':0.025,'mbp_ni':0.025,'creatinine':0.02,'wbc':0.02}
    for i, f in enumerate(FEATURE_NAMES):
        if f in boost: w[i] += boost[f] * score
    w /= w.sum()
    top = np.argsort(w)[::-1][:n]
    return [{'name': FEATURE_NAMES[j], 'weight': round(float(w[j]), 4)} for j in top]
def gen_vitals(s, m):
    c = (s + m) / 2
    return {'hr':int(np.clip(72+c*48+np.random.normal(0,10),40,170)),
            'sbp':int(np.clip(125-c*40+np.random.normal(0,12),55,190)),
            'dbp':int(np.clip(75-c*25+np.random.normal(0,8),30,120)),
            'spo2':int(np.clip(98-c*10+np.random.normal(0,1.5),80,100)),
            'temp':round(float(np.clip(37.0+c*2.8+np.random.normal(0,0.35),35,41.5)),1),
            'rr':int(np.clip(14+c*18+np.random.normal(0,2.5),8,45)),
            'map':int(np.clip(90-c*30+np.random.normal(0,8),40,130)),
            'gcs':int(np.clip(15-c*8+np.random.normal(0,1.5),3,15))}
def gen_labs(s, m):
    c = s*0.6 + m*0.4
    return {'wbc':round(float(np.clip(7.5+c*15+np.random.normal(0,2),0.5,35)),1),
            'lactate':round(float(np.clip(0.8+c*8+np.random.normal(0,0.5),0.5,15)),1),
            'creatinine':round(float(np.clip(0.9+c*4+np.random.normal(0,0.3),0.4,12)),1),
            'bilirubin':round(float(np.clip(0.8+c*6+np.random.normal(0,0.4),0.2,20)),1),
            'platelets':int(np.clip(250-c*180+np.random.normal(0,30),20,450)),
            'inr':round(float(np.clip(1.1+c*2.5+np.random.normal(0,0.2),0.9,8)),1),
            'sodium':int(np.clip(138+c*8+np.random.normal(0,4),120,160)),
            'potassium':round(float(np.clip(4.0+c*1.5+np.random.normal(0,0.4),2.5,7.5)),1),
            'ph':round(float(np.clip(7.40-c*0.18+np.random.normal(0,0.02),6.9,7.55)),2),
            'pco2':round(float(np.clip(40+c*15+np.random.normal(0,4),20,80)),1),
            'po2':round(float(np.clip(95-c*40+np.random.normal(0,8),40,150)),1),
            'bicarbonate':round(float(np.clip(24-c*8+np.random.normal(0,2),8,35)),1),
            'base_excess':round(float(np.clip(-1-c*10+np.random.normal(0,1.5),-20,5)),1),
            'glucose':int(np.clip(110+c*80+np.random.normal(0,20),50,450)),
            'albumin':round(float(np.clip(3.8-c*1.8+np.random.normal(0,0.3),1.2,5)),1),
            'bun':round(float(np.clip(15+c*50+np.random.normal(0,5),5,120)),1)}
sep_base  = np.random.beta(2, 3.5, N)
mort_base = np.random.beta(1.8, 4.5, N)
mort_base = np.clip(mort_base + np.random.normal(0, 0.08, N)*(sep_base-0.37), 0.02, 0.97)
sep_labels  = (sep_base > 0.45).astype(int)
mort_labels = (mort_base > 0.50).astype(int)
for i in range(N):
    if sep_labels[i]:  sep_base[i]  = np.clip(sep_base[i]*1.4+0.15+np.random.normal(0,0.05),0.4,0.97)
    if mort_labels[i]: mort_base[i] = np.clip(mort_base[i]*1.5+0.12+np.random.normal(0,0.05),0.35,0.97)
ages=np.random.randint(22,92,N); sexes=np.random.choice(['M','F'],N)
hours_icu=np.random.randint(1,120,N); diagnoses=np.random.choice(DIAGNOSES,N)
patients=[]
for i in range(N):
    s=float(np.clip(sep_base[i],0.02,0.97)); m=float(np.clip(mort_base[i],0.02,0.97))
    sev=(s+m)/2
    patients.append({'stay_id':int(100000+i*37+np.random.randint(0,100)),
        'age':int(ages[i]),'sex':sexes[i],'hours_in_icu':int(hours_icu[i]),
        'admission_dx':diagnoses[i],'sepsis_score':round(s,4),'sepsis_label':int(sep_labels[i]),
        'sepsis_trajectory':gen_traj(s,sep_labels[i]),'sepsis_top_feats':gen_feats(s),
        'mortality_score':round(m,4),'mortality_label':int(mort_labels[i]),
        'mortality_trajectory':gen_traj(m,mort_labels[i]),'mortality_top_feats':gen_feats(m),
        'overall_risk':round(float(s*0.55+m*0.45),4),
        'vitals':gen_vitals(s,m),'labs':gen_labs(s,m),
        'on_vasopressors':bool(sev>0.55 and np.random.random()>0.35),
        'on_ventilator':bool(sev>0.60 and np.random.random()>0.40),
        'on_antibiotics':bool(s>0.40 and np.random.random()>0.25),
        'on_crrt':bool(m>0.65 and np.random.random()>0.55),
        'on_heparin':bool(np.random.random()>0.65),
        'sepsis_delta':round(float(np.random.normal(0,0.018)+(0.006 if sep_labels[i] else -0.004)),3),
        'mortality_delta':round(float(np.random.normal(0,0.015)+(0.005 if mort_labels[i] else -0.003)),3),
        'seq_len':int(min(hours_icu[i],24)),'bed':BEDS[i] if i<20 else None})
patients.sort(key=lambda x:x['overall_risk'],reverse=True)
for i,p in enumerate(patients[:20]): p['bed']=BEDS[i]
for p in patients[20:]: p['bed']=None
with open('/content/prism/synthetic_patients.json','w') as f: json.dump(patients,f)
print(f'Generated {N} patients')
print(f'  Sepsis +ve:   {sum(p["sepsis_label"] for p in patients)} ({sum(p["sepsis_label"] for p in patients)/N:.1%})')
print(f'  Mortality +ve:{sum(p["mortality_label"] for p in patients)} ({sum(p["mortality_label"] for p in patients)/N:.1%})')
print('  Saved: /content/prism/synthetic_patients.json')

Generated 200 patients
  Sepsis +ve:   53 (26.5%)
  Mortality +ve:29 (14.5%)
  Saved: /content/prism/synthetic_patients.json


In [3]:
# Cell 3 — Write dashboard using base64 (avoids all escaping issues)
import base64, os, ast
os.makedirs('/content/prism', exist_ok=True)
_b64 = '...'
dashboard_code = base64.b64decode(_b64).decode('utf-8')
with open('/content/prism/app.py', 'w') as f:
    f.write(dashboard_code)
with open('/content/prism/requirements.txt', 'w') as f:
    f.write('streamlit>=1.32.0\nplotly>=5.18.0\npandas>=2.0.0\nnumpy>=1.24.0\n')
try:
    ast.parse(dashboard_code)
    print(f'app.py syntax OK ({len(dashboard_code.splitlines())} lines)')
except SyntaxError as e:
    print(f'Syntax error: {e}')
print('Files:', os.listdir('/content/prism'))

# Write Streamlit dark theme config so dataframe background matches dashboard
import os
os.makedirs('/content/prism/.streamlit', exist_ok=True)
with open('/content/prism/.streamlit/config.toml', 'w') as f:
    f.write('[theme]\n')
    f.write('base = "light"\n')
    f.write('backgroundColor = "#f8fafc"\n')
    f.write('secondaryBackgroundColor = "#ffffff"\n')
    f.write('textColor = "#1e293b"\n')
    f.write('primaryColor = "#e53e3e"\n')
    f.write('base = "dark"\n')
    f.write('backgroundColor = "#0f1117"\n')
    f.write('secondaryBackgroundColor = "#141820"\n')
    f.write('textColor = "#e2e8f0"\n')
    f.write('primaryColor = "#fc8181"\n')


app.py syntax OK (510 lines)
Files: ['synthetic_patients.json', 'app.py', 'requirements.txt']


In [4]:
# Cell 4 — Launch dashboard
import subprocess, time
from google.colab.output import eval_js
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/prism/app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
     '--server.address', '0.0.0.0'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/content/prism')
print('Starting', end='')
for _ in range(10): time.sleep(1); print('.', end='', flush=True)
print()
import urllib.request
try:
    urllib.request.urlopen('http://localhost:8501', timeout=3)
    print('Streamlit is running')
except: print('Server still warming up — try the URL anyway')
url = eval_js('google.colab.kernel.proxyPort(8501)')
print()
print('=' * 60)
print('  PRISM Dashboard is live')
print()
print('  Open this URL in Chrome:')
print(f'  {url}')
print()
print('  Right-click the URL and copy it')
print('  Must be logged into Google in that browser')
print('  Keep this Colab tab open')
print('=' * 60)

Starting..........
Streamlit is running

  PRISM Dashboard is live

  Open this URL in Chrome:
  https://8501-m-s-kkb-use1d0-2nh3q6hxdffx7-d.us-east1-0.prod.colab.dev

  Right-click the URL and copy it
  Must be logged into Google in that browser
  Keep this Colab tab open


In [ ]:
# Cell 5 — Live simulation (3-second updates)
# Writes updated scores to synthetic_patients.json every 3 seconds.
# The dashboard re-reads the file on each rerun cycle.
# Stop with the interrupt (square) button.
import json, time, os, numpy as np

DATA_FILE = '/content/prism/synthetic_patients.json'
TMP_FILE  = DATA_FILE + '.tmp'

with open(DATA_FILE) as f:
    patients = json.load(f)

# Store base scores for mean reversion
for p in patients:
    p['_base_sep']  = p['sepsis_score']
    p['_base_mort'] = p.get('mortality_score', p['sepsis_score'])

BEDS = [f'ICU-{i+1:02d}' for i in range(20)]
print(f'Simulating {len(patients)} patients — press Stop button to halt')
print('Dashboard updates every 3 seconds automatically')
print()
update = 0
try:
    while True:
        update += 1
        for p in patients:
            for key, bk, lk in [
                ('sepsis_score',    '_base_sep',  'sepsis_label'),
                ('mortality_score', '_base_mort', 'mortality_label'),
            ]:
                model  = key.split('_')[0]
                bias   =  0.004 if p.get(lk) else -0.004
                noise  = float(np.random.normal(0, 0.015))
                revert = (p[bk] - p[key]) * 0.10
                new    = float(np.clip(p[key] + bias + noise + revert, 0.02, 0.97))
                p[model + '_delta'] = round(new - p[key], 3)
                p[key] = round(new, 4)
                traj = p.get(model + '_trajectory', [])
                if traj:
                    traj.pop(0)
                    traj.append(round(new, 4))
            # Drift vitals slightly
            v = p.get('vitals', {})
            if v:
                v['hr']   = int(np.clip(v['hr']   + np.random.randint(-3, 4),  40, 170))
                v['sbp']  = int(np.clip(v['sbp']  + np.random.randint(-4, 5),  55, 190))
                v['spo2'] = int(np.clip(v['spo2'] + np.random.randint(-1, 2),  80, 100))
                v['rr']   = int(np.clip(v['rr']   + np.random.randint(-2, 3),   8,  45))
            p['overall_risk'] = round(
                p['sepsis_score'] * 0.55 + p.get('mortality_score', p['sepsis_score']) * 0.45, 4)

        # Re-sort by overall risk
        patients.sort(key=lambda x: x['overall_risk'], reverse=True)
        for i, p in enumerate(patients[:20]): p['bed'] = BEDS[i]
        for p in patients[20:]:               p['bed'] = None

        # Atomic write — write to .tmp then rename so dashboard never reads a partial file
        with open(TMP_FILE, 'w') as f:
            json.dump(patients, f)
        os.replace(TMP_FILE, DATA_FILE)

        top = patients[0]
        print(f'Update {update:04d} | Top: {top["bed"]} '
              f'sep={top["sepsis_score"]:.3f} mort={top.get("mortality_score",0):.3f} '
              f'overall={top["overall_risk"]:.3f}', end='\r')
        time.sleep(5)
except KeyboardInterrupt:
    print(f'\nSimulation stopped after {update} updates')

Simulating 200 patients — press Stop button to halt
Dashboard updates every 3 seconds automatically



In [ ]:
# Cell 6 — Save to Drive + download zip
import shutil, os
from google.colab import drive, files
try: drive.mount('/content/drive', force_remount=False)
except: pass
save_dir = '/content/drive/MyDrive/prism_dashboard'
os.makedirs(save_dir, exist_ok=True)
for fn in ['app.py', 'synthetic_patients.json', 'requirements.txt']:
    shutil.copy(f'/content/prism/{fn}', f'{save_dir}/{fn}')
    print(f'  Saved {fn}')
shutil.make_archive('/content/prism_icu', 'zip', '/content', 'prism')
files.download('/content/prism_icu.zip')
print('Downloaded prism_icu.zip')